# Droplet rebound simulation - Postprocessing

This notebook is used to postprocess the simulations for the droplet rebound test case.

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
// BoSSSshell.WorkflowMgm.Init("DropletRebound");
// var sessions = BoSSSshell.WorkflowMgm.Sessions;

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletRebound");
var sessions = databases.Pick(0).Sessions;

In [ ]:
sessions

#0: DropletRebound	DropletRebound_8x8x8AMR1_dropVelocity0*	11/02/2023 20:59:59	77de991d...
#1: DropletRebound	DropletRebound_8x8x8AMR1_dropVelocity10vH_noGravity*	11/02/2023 20:51:00	a97b046c...
#2: DropletRebound	DropletRebound_8x8x16AMR1_dropVelocity0_noGravity_restart*	10/30/2023 14:28:06	a463354f...
#3: DropletRebound	DropletRebound_8x8x16AMR1_dropVelocity0_noGravity	10/26/2023 14:44:00	2ea98434...
#4: DropletRebound	DropletRebound_8x8x16AMR2_Picard4_1e-6*	10/20/2023 13:36:33	7250bb76...
#5: DropletRebound	DropletRebound_8x8x16AMR2_Picard4_5e-6*	10/20/2023 13:35:49	7a0b7406...
#6: DropletRebound	DropletRebound_8x8x16AMR2_Picard4*	10/16/2023 14:59:17	8a9e4f25...
#7: DropletRebound	DropletRebound_8x8x16AMR2_Picard4*	10/16/2023 14:54:45	47629bb3...
#8: DropletReboundInit	DropletRebound_8x8x16AMR2_Picard4_InitState	09/01/2023 14:54:52	38753b53...
#9: DropletRebound	DropletRebound-TestRunRampUpInitDropVelocity3*	08/25/2023 15:30:07	3a1b5373...
#10: DropletRebound	DropletRebound-TestRunR

In [ ]:
var sess = sessions.Pick(0);
sess

DropletRebound	DropletRebound_8x8x16AMR1_dropVelocity0_noGravity	10/26/2023 14:44:00	2ea98434...

In [ ]:
sess.Timesteps//.Skip(2).Pick(0)

#0:  { Time-step: 0.0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#1:  { Time-step: 0.1; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#2:  { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#3:  { Time-step: 1; Physical time: 1E-06s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Resi

In [ ]:
//sess.KeysAndQueries["Option_LevelSetEvolution"]
sess.GetSessionDirectory()

\\hpccluster\hpccluster-scratch\smuda\DropletRebound\sessions\7a0b7406-9445-4ea4-9bb9-2491e513ae16

In [ ]:
sess.Timesteps.Skip(2).Every(20).Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound__DropletRebound-TestRunTransient6__745e7570-dcb9-450f-848f-7d6de791574a


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound__DropletRebound-TestRunTransient6__745e7570-dcb9-450f-848f-7d6de791574a

## Droplet metrics (center of mass, volume, sphericity) over time

In [ ]:
using BoSSS.Foundation.Quadrature;

In [ ]:
var sessTimesteps = sess.Timesteps.Skip(2).Every(10);
//sessTimesteps

In [ ]:
int timesteps = sessTimesteps.Count();
timesteps

11

In [ ]:
double[] time = new double[timesteps];
MultidimensionalArray centerOfMass = MultidimensionalArray.Create(timesteps, 3);
double[] volume = new double[timesteps];
double[] sphericity = new double[timesteps];

In [ ]:
int degPhi = sessTimesteps.ElementAt(0).GetField("Phi").Basis.Degree;
int quadOrder = (2 * degPhi) * 2 + 1;   // using Saye

In [ ]:
for (int ts = 0; ts < timesteps; ts++) {
    var tStep = sessTimesteps.ElementAt(ts);

    time[ts] = tStep.PhysicalTime;

    SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
    GridData grdData = (GridData)phi.GridDat; 
    LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
    levelSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
    LsTrk.UpdateTracker(tStep.PhysicalTime);

    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), quadOrder, 1).XQuadSchemeHelper;
    SpeciesId spcId = LsTrk.GetSpeciesId("A");
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcId);


    // volume
    double dropVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            EvalResult.SetAll(1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
            dropVolume += ResultsOfIntegration[i, 0];
        }
    ).Execute();

    volume[ts] = dropVolume;


    // center of mass
    double[] dropCoM = new double[3];
    CellQuadrature.GetQuadrature(new int[] { 3 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            NodeSet nodes_global = QR.Nodes.CloneAs();
            for (int i = i0; i < i0 + Length; i++) {
                LsTrk.GridDat.TransformLocal2Global(QR.Nodes, nodes_global, i);
                EvalResult.AccSubArray(1.0, nodes_global, new int[] { i - i0, -1, -1 });
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int d = 0; d < 3; d++) {
                for(int i = 0; i < Length; i++)
                    dropCoM[d] += ResultsOfIntegration[i, d];
            }
        }
    ).Execute();

    for(int d = 0; d < 3; d++) {
        centerOfMass[ts,d] = dropCoM[d] / dropVolume;
    }


    // // sphericity
    // double dropSurface = 0.0;
    // CellQuadratureScheme cqs = SchemeHelper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask());
    // CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    //     cqs.Compile(LsTrk.GridDat, quadOrder),
    //     delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
    //         EvalResult.SetAll(1.0);
    //     },
    //     delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
    //         for (int i = 0; i < Length; i++)
    //         dropSurface += ResultsOfIntegration[i, 0];
    //     }
    // ).Execute();

    // sphericity[ts] = Math.Pow(Math.PI, 1.0/3.0) * Math.Pow(6*dropVolume, 2.0/3.0) / dropSurface;

}

In [ ]:
centerOfMass

Dimension,Lengths,Storage,IsContinious,Length,NoOfCols,NoOfRows,IsLocked
2,"[ 11, 3 ]","[ 0.07800000000000004, -4.239196052665471E-21, 0.0007499974787539876, 0.07800000171292445, 2.301191546799821E-09, 0.0007499944642057987, 0.07800000377516782, 4.87269840556686E-09, 0.0007499911521975812, 0.078000006215408, 8.048066927573437E-09, 0.0007499884405554919, 0.07800000898066191, 1.1632952263636328E-08, 0.0007499862706424031, 0.07800001201356734, 1.5511994736025555E-08, 0.0007499841325155639, 0.07800001520932696, 1.9661215529071736E-08, 0.0007499818540504049, 0.07800001866704068, 2.412837057232821E-08, 0.0007499791805228088, 0.07800002230421403, 2.885837628227773E-08, 0.0007499761213760117, 0.07800002614180436, 3.3898881815141697E-08, 0.0007499727147213897, 0.07800003015150812, 3.912394163906895E-08, 0.0007499690200601223 ]",True,33,3,11,False


In [ ]:
var fmt = new PlotFormat();    
fmt.LineColor = LineColors.Blue;

var gp = new Gnuplot();
gp.SetMultiplot(1,2);

Plot2Ddata plt = new Plot2Ddata(); 
plt.AddDataGroup("centerOfMassZ", time, centerOfMass.ExtractSubArrayShallow(-1,2).To1DArray(), fmt);
gp.SetSubPlot(0, 0, "centerOfMassZ");
plt.ToGnuplot(gp);

// plt = new Plot2Ddata(); 
// plt.AddDataGroup("sphericity", time, sphericity, fmt);
// gp.SetSubPlot(0, 1, "sphericity");
// plt.ToGnuplot(gp);

//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:500)

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.00074997 
 
 
 
 
 0.00074997 
 
 
 
 
 0.00074997 
 
 
 
 
 0.00074998 
 
 
 
 
 0.00074998 
 
 
 
 
 0.00074999 
 
 
 
 
 0.00074999 
 
 
 
 
 0.00075000 
 
 
 
 
 0 
 
 
 
 
 1x10 -5 
 
 
 
 
 2x10 -5 
 
 
 
 
 3x10 -5 
 
 
 
 
 4x10 -5 
 
 
 
 
 5x10 -5 
 
 
 
 
 6x10 -5 
 
 
 
 
 7x10 -5 
 
 
 
 
 8x10 -5 
 
 
 
 
 9x10 -5 
 
 
 
 
 0.0001 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 centerOfMassZ 
 
 
 centerOfMassZ

## mean vertical velocity / lift over time

In [ ]:
double[] time = new double[timesteps];
double[] meanVertVelocity = new double[timesteps];
// double[] lift = new double[timesteps];
MultidimensionalArray pressureForce = MultidimensionalArray.Create(timesteps, 3);

In [ ]:
int degPhi = sess.Timesteps.ElementAt(0).GetField("Phi").Basis.Degree;
int quadOrder = (2 * degPhi) * 2 + 1;   // using Saye

In [ ]:
for (int ts = 0; ts < timesteps; ts++) {
    var tStep = sessTimesteps.ElementAt(ts);

    time[ts] = tStep.PhysicalTime;

    SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
    GridData grdData = (GridData)phi.GridDat; 
    LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
    levelSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
    LsTrk.UpdateTracker(tStep.PhysicalTime);

    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), quadOrder, 1).XQuadSchemeHelper;
    SpeciesId spcId = LsTrk.GetSpeciesId("A");
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcId);


    // volume
    double dropVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            EvalResult.SetAll(1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
                dropVolume += ResultsOfIntegration[i, 0];
        }
    ).Execute();


    // mean vertical velocity
    double vertVel = 0.0;
    DGField velocityZ = ((XDGField)tStep.GetField("VelocityZ")).GetSpeciesShadowField("A");
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            velocityZ.Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1,-1,0), 1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
                vertVel += ResultsOfIntegration[i, 0];
        }
    ).Execute();

    meanVertVelocity[ts] = vertVel / dropVolume;


    // // lift
    // double[] pForce = new double[3];
    // DGField pressure = ((XDGField)tStep.GetField("Pressure")).GetSpeciesShadowField("B");
    // CellQuadratureScheme cqs = SchemeHelper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask());
    // CellQuadrature.GetQuadrature(new int[] { 3 }, LsTrk.GridDat,
    //     cqs.Compile(LsTrk.GridDat, quadOrder),
    //     delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
    //         var normals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, Length);
    //         int K = EvalResult.GetLength(1);
    //         var pressureEval = MultidimensionalArray.Create(Length, K);
    //         pressure.Evaluate(i0, Length, QR.Nodes, pressureEval, 1.0);  
    //         for(int d = 0; d < 3; d++) {              
    //             for (int j = 0; j < Length; j++) {
    //                 for (int k = 0; k < K; k++) {
    //                     EvalResult[j, k, d] = pressureEval[j, k] * normals[j, k, d];
    //                 }
    //             }
    //         }
                
    //     },
    //     delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
    //         for(int d = 0; d < 3; d++) {  
    //             for (int i = 0; i < Length; i++)
    //                 pForce[d] += ResultsOfIntegration[i, d];
    //         }
    //     }
    // ).Execute();

    // for(int d = 0; d < 3; d++)
    //     pressureForce[ts, d] = pForce[d];

}

In [ ]:
var fmt = new PlotFormat();    
fmt.LineColor = LineColors.Blue;

var gp = new Gnuplot();
gp.SetMultiplot(1,2);

Plot2Ddata plt = new Plot2Ddata(); 
plt.AddDataGroup("meanVelocityZ", time, meanVertVelocity, fmt);
gp.SetSubPlot(0, 0, "meanVelocityZ");
plt.ToGnuplot(gp);

plt = new Plot2Ddata(); 
plt.AddDataGroup("lift", time, pressureForce.ExtractSubArrayShallow(-1,2).To1DArray(), fmt);
gp.SetSubPlot(0, 1, "lift");
plt.ToGnuplot(gp);

//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:500)